- rkl or fkl，以及 why fkl-topk
- 如何 reuse rl framework
- verl / slime details

### FKL vs. RKL

- 设目标分布 $p$，待优化模型分布是 $q_\theta$，
- forward kl：$D_{\mathrm{KL}}(p \| q_\theta)= \sum_x p(x)\log \frac{p(x)}{q_\theta(x)}$
    - $D_{\mathrm{KL}}(p \| q_\theta)= \sum_x p(x)\log p(x) - \sum_x p(x)\log q_\theta(x)$
    - $D_{\mathrm{KL}}(p \| q_\theta)= H(p, q_\theta) - H(p)$
    - 因此：$\arg\min_\theta D_{\mathrm{KL}}(p \| q_\theta)=\arg\min_\theta H(p, q_\theta)$
        - fkl 等价于交叉熵损失
        - $\nabla_\theta D_{\mathrm{KL}}(p \| q_\theta)=\nabla_\theta H(p, q_\theta)=-\mathbb{E}_{x\sim p}\left[\nabla_\theta \log q_\theta(x)\right]$
    - 具体到分类任务，如果标签是 one-hot，例如真实类别是 $y$，交叉熵就是：$-\log q_\theta(y)$（negative log-likelihood loss）
- 若优化的是 RKL：$D_{\mathrm{KL}}(q_\theta \| p)=\mathbb{E}_{q_\theta}[\log q_\theta(x)] - \mathbb{E}_{q_\theta}[\log p(x)]$
    - $D_{\mathrm{KL}}(q_\theta \| p)=\sum_x q_\theta(x)\left[\log q_\theta(x)-\log p(x)\right]$
    - $D_{\mathrm{KL}}(q_\theta \| p)=-H(q_\theta)-\mathbb{E}_{q_\theta}[\log p(x)]$
    - 直觉上，RKL 评估的是，模型自己采样出来的 $x$，在模型 $q_\theta$ 里有多合理，在目标 $p$ 里又有多合理。
        - 如果 $q_\theta(x)\gt 0$，但 $p(x)=0$，则 $\log p(x)=-\infty$，RKL 直接到无穷大；

看一个离散例子，假设：$P=(0.5,0.5),Q=(0.9,0.1)$，
那么：

$$
D_{\mathrm{KL}}(P\|Q)
=0.5\log\frac{0.5}{0.9}
+0.5\log\frac{0.5}{0.1}
\approx0.511
$$

FKL 中两个位置都由 $P$ 赋予 0.5 的权重，所以 Q 严重低估第二个位置会受到明显惩罚。

反过来：
$$
D_{\mathrm{KL}}(Q\|P)=0.9\log\frac{0.9}{0.5}+0.1\log\frac{0.1}{0.5}
\approx0.368
$$

RKL 中第二个位置只有 $Q$ 给出的 0.1 权重。因为 $Q$ 很少访问那里，所以忽略它受到的惩罚相对较小。

----

假设：$P=(0.9,0.1),Q=(0.5,0.5)$，
那么：

$$
D_{\mathrm{KL}}(P\|Q)=0.9\log\frac{0.9}{0.5}+0.1\log\frac{0.1}{0.5}
\approx0.368
$$

$$
D_{\mathrm{KL}}(Q\|P)=
=0.5\log\frac{0.5}{0.9}
+0.5\log\frac{0.5}{0.1}
\approx0.511
$$

In [2]:
import torch
import torch.nn.functional as F
from torch.distributions import Categorical, kl_divergence

P = torch.tensor([0.5, 0.5])
Q = torch.tensor([0.9, 0.1])

# ---------- 方案 1: torch.distributions（参数顺序与数学记号一致，推荐）----------
p, q = Categorical(probs=P), Categorical(probs=Q)
fkl_d = kl_divergence(p, q)   # KL(P||Q)
rkl_d = kl_divergence(q, p)   # KL(Q||P)

# ---------- 方案 2: F.kl_div（注意：参数顺序是反的！）----------
# F.kl_div(input=log q, target=p) == KL(p || q)
fkl_f = F.kl_div(Q.log(), P, reduction="sum")
rkl_f = F.kl_div(P.log(), Q, reduction="sum")

def kl(measure, other):
    """KL(measure || other) = sum_i measure_i * log(measure_i / other_i)"""
    return (measure * (measure.log() - other.log())).sum()

| | Reverse $D(q\|p)$ | Forward $D(p\|q)$ |
|---|---|---|
| 惩罚项 | $q\log(q/p)$，罚"教师不认可但学生敢说"（zero-forcing） | $p\log(p/q)$，罚"教师会说但学生不敢"（zero-avoiding） |
| 行为 | **mode-seeking**：学生收敛到教师的一个子模式，锐化 | **mean-covering**：学生摊开去覆盖教师全部行为 |
| 容量失配时 | 8B 只学 32B 的一个模式，学得干净 | 8B 被迫摊平概率质量，变得犹豫、退化成平均 |

- FKL: $p\text{ 在哪里有质量，}q_\theta\text{ 就必须去覆盖哪里。}$
- RKL: $q_\theta\text{ 把质量放在哪里，那里必须是 }p\text{ 认可的高概率区域。}$
- FKL 更容易“覆盖多个可能答案”，RKL 更容易“选择一个高置信答案”。在生成模型、变分推断、RL policy optimization、蒸馏里，这个差异会直接体现为：FKL 偏保守、覆盖；RKL 偏尖锐、挑模式。
- OPD 的典型设定就是小学生←大教师（8B←32B、4B←35B-A3B）；
    - 对于数学推理这类任务本来就只需要一条对的路径，不需要覆盖教师的全部多样性。**reverse KL 天然匹配这个场景**，同时又恰好是 on-policy 管道唯一能低成本估计的方向——目标的选择和工程的可行性在这里是同一个答案，这不是巧合，是 on-policy distillation 这个方法能成立的原因。

#### why rkl

- KL 的测度（用哪个分布作为测度（measure））在第一个参数上

    $$D_{KL}(a\|b)=\sum_{v}a(v)\log\frac{a(v)}{b(v)}=\mathbb{E}_{v\sim a}\!\left[\log a(v)-\log b(v)\right]$$

    - **期望是对第一个参数取的。** 从谁那里采样，就只能估计以谁打头的那个 KL。
    - on-policy 的设定是 $y_t\sim q_t$（学生自己的 rollout，$q_t=\pi_\theta(\cdot\mid y_{<t})$，$p_t=\pi_{\text{teacher}}(\cdot\mid y_{<t})$）：
    - **reverse KL** $D(q_t\|p_t)=\mathbb{E}_{v\sim q_t}[\log q_t-\log p_t]$ —— 采样分布和外层测度**是同一个**。$\log q_t(y_t)-\log p_t(y_t)$ 直接就是它的单样本无偏估计（k1）。
    - **forward KL** $D(p_t\|q_t)=\mathbb{E}_{v\sim p_t}[\log p_t-\log q_t]$ —— 计算的是**教师分布下的期望**，而你手里的样本来自学生。测度不匹配，必须换测度。

#### fkl topk

- 可以重要性采样换测度：

    $$D_{KL}(p_t\|q_t)=\mathbb{E}_{v\sim q_t}\!\left[\frac{p_t(v)}{q_t(v)}\log\frac{p_t(v)}{q_t(v)}\right]=\mathbb{E}_{v\sim q_t}\big[w\log w\big],\qquad w=e^{\log p_t(y_t)-\log q_t(y_t)}$$
    
    - $w\log w$ **只用那两个标量就能算**，而且无偏。问题在于二阶矩：

    $$\mathbb{E}_{q}\big[w^2\log^2 w\big]=\sum_v \frac{p_t(v)^2}{q_t(v)}\log^2\frac{p_t(v)}{q_t(v)}$$

    - 分母上那个 $q_t(v)$ 是致命的。forward KL 的**信号本身**恰恰集中在 $p$ 大、$q$ 小的 token 上（这正是 zero-avoiding / mean-covering 的定义）——也就是学生几乎不采的那些 token。于是：

> **信号所在的地方，正是采样密度趋于零的地方。**

- 所以 per-token 解析 KL 要的是"分布"，不是"标量"

任何一个方向的**解析** per-token KL，都要在词表上求和，需要 $O(|V|)$ 个数（Qwen3 是 15 万量级）。传输和通信的问题，所以退而求其次：

$$D_{KL}(p_t\|q_t)\approx\sum_{v\in\mathrm{top}\text{-}k(p_t)}p_t(v)\log\frac{p_t(v)}{q_t(v)}$$

top-k 截断之所以对 **forward** KL 特别合理：外层权重是 $p_t(v)$，教师的 top-64 通常已经覆盖了它 99%+ 的质量，被截掉的尾巴权重本来就小。反过来对 reverse KL 用 top-k 就不自然了——外层权重是 $q_t(v)$（学生的），按教师的 top-k 去截学生的分布是错位的。

这解释了 verl 的接口设计：`forward_kl_topk` 是唯一标了 `use_topk=True` 的 loss，必须拿到 `teacher_logprobs` + `teacher_ids` 两个 $(bsz, seqlen, 64)$ 的张量（$\widetilde\ell_t(\theta;s_t)
=\sum_{v\in K_t}p_t(v\mid s_t)\left[\log p_t(v\mid s_t)-\log q_\theta(v\mid s_t)\right]$，teacher 已经显式传来了 $K_t$ 中每个 token 的概率和 ID，student 对这些 ID 做 gather：）；而 `k1/k2/k3/low_var_kl/abs/mse` 全部走 `use_estimator=True`，只要采样 token 上的一个标量。**接口的形状是被 KL 的方向逼出来的。**

- verl 的 `forward_kl_topk + use_policy_gradient=False` 则是在学生前缀上直接反传 teacher-top-k loss：
    - $y_t \sim q_{\text{rollout}}(\cdot\mid s_t), \quad s_t=(x,y_{<t})$
    - 记 $\mathcal K_t=\text{top-}k\big(p_t(\cdot\mid y_{<t})\big)$ 是教师在位置 $t$ 的 top-k token 集合（代码里的 teacher_topk_ids，$k$ 默认 64）。逐 token 损失是：


$$
\ell_t(\theta)=\sum_{v\in\mathcal{K}_t}p_t(v)\left[\log p_t(v)-\log q_\theta\!\left(v\mid y_{<t}\right)\right]
$$

- forward_kl_topk 应配 use_policy_gradient=False；若设为 true (`advantages = -distillation_losses.detach()`)，verl 自己会警告 top-k 的非采样 token 信号基本没被利用：
    - 在同一个生成位置 $t$，teacher 给出的 top-k 候选中，没有被 student rollout 实际选为 $y_t$ 的那些 token。

```python
if self.use_policy_gradient and self.loss_mode == "forward_kl_topk":
    print("WARNING: forward_kl_topk is most effective as a supervised distillation loss (use_policy_gradient=False). With policy gradient, the update uses only the sampled token's logprob ∇logπ(a), so the top-k distributional signal (how non-sampled logits should move) is largely unused.")
```

### reuse RL

> 最小化 reverse KL 等价于一种 RL 最大化问题（reverse KL 的梯度就是 policy gradient）

$$
\begin{split}
\nabla_{\theta}\mathcal{L}_{RL} &= - \mathbb{E}_{y \sim \pi_{\theta}} \left[ \nabla_{\theta}\log\pi_{\theta}(y) \cdot A(y) \right]\\
\mathcal{L}_{OPD}(\theta) &= D_{KL}(\pi_{\theta} || \pi_{E}) = \sum_{y} \pi_{\theta}(y) \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)}
\end{split}
$$
- $\nabla_{\theta} D_{KL}(\pi_{\theta} || \pi_{E}) = \sum_{y} \left( \nabla_{\theta}\pi_{\theta}(y) \cdot \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)} + \pi_{\theta}(y) \cdot \nabla_{\theta} \left[ \log \pi_{\theta}(y) - \log \pi_{E}(y) \right] \right)$
    - 右侧部分 $\pi_{\theta}(y) \cdot \nabla_{\theta}\log\pi_{\theta}(y) \rightarrow \sum_{y} \nabla_{\theta}\pi_{\theta}(y) = \nabla_{\theta} \left( \sum_{y} \pi_{\theta}(y) \right)=0$
- $\nabla_{\theta} D_{KL} = \sum_{y} \nabla_{\theta}\pi_{\theta}(y) \cdot \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)}=\sum_{y} \pi_{\theta}(y) \nabla_{\theta}\log\pi_{\theta}(y) \cdot \log \frac{\pi_{\theta}(y)}{\pi_{E}(y)}= - \mathbb{E}_{y \sim \pi_{\theta}} \left[ \nabla_{\theta}\log\pi_{\theta}(y) \cdot \log \frac{\pi_{E}(y)}{\pi_{\theta}(y)} \right]$
- 定义：$A(y) = \log \frac{\pi_{E}(y)}{\pi_{\theta}(y)}$
    - $A_t = \text{sg}\left[\log\frac{\pi_{E_i}(y_t|x, y_{<t})}{\pi_\theta(y_t|x, y_{\le t})}\right]$

$$
\log\frac{\pi_T(a_t|s_t)}{\pi_\theta(a_t|s_t)}=\log\pi_T(a_t|s_t)-\log\pi_\theta(a_t|s_t).
$$

这在代码里作为 reward/advantage 使用。对应的 loss 记成相反数：

$$
d_t=\log\pi_\theta(a_t|s_t)-\log\pi_T(a_t|s_t).
$$

- 如果 student 给某 token 很高概率，但 teacher 给低概率，那么：$\log\pi_\theta-\log\pi_T > 0$ loss 高，应该压低它。
- 如果 teacher 比 student 更认可这个 sampled token，那么：$\log\pi_\theta-\log\pi_T < 0$ 它变成正向学习信号。

### seq level

- $\log\frac{q(y_{1:T})}{p(y_{1:T})}=\log\prod_{t=1}^{T}\frac{q_t(y_t\mid y_{<t})}{p_t(y_t\mid y_{<t})}=\sum_{t=1}^{T}\log\frac{q_t(y_t\mid y_{<t})}{p_t(y_t\mid y_{<t})}$

  
$$D_{\mathrm{KL}}(q\Vert p)
=\mathbb E_{Y_{1:T}\sim q}
\left[
\log\frac{q(Y_{1:T})}{p(Y_{1:T})}
\right]=\mathbb E_{Y_{1:T}\sim q}\left[\sum_{t=1}^{T}\log\frac{q_t(Y_t\mid Y_{<t})}{p_t(Y_t\mid Y_{<t})}\right]$$


- 序列级 reverse KL，$q(y_{1:T})=\prod_t q_t(y_t\mid y{<t})$，第 $t$ 项只依赖 $y_{\le t}$，把那个大期望按位置展开：
$$
D_{\mathrm{KL}}\!\left(q(Y_{1:T})\Vert p(Y_{1:T})\right)=\sum_{t=1}^{T}\underbrace{\mathbb{E}_{Y_{<t}\sim q}}_{\text{外层：前缀状态分布}}\!\left[\underbrace{\mathbb{E}_{Y_t\sim q_t(\cdot\mid Y_{<t})}\!\left[\log\frac{q_t(Y_t\mid Y_{<t})}{p_t(Y_t\mid Y_{<t})}\right]}_{\text{内层：}D_{\mathrm{KL}}\left(q_t(\cdot\mid Y_{<t})\Vert p_t(\cdot\mid Y_{<t})\right)}\right].
$$
  - 外层 $\mathbb{E}_{y{<t}\sim\cdot}$ —— 前缀（也就是"状态"）从谁的分布里来。谁在自回归地生成这条轨迹。
  - 内层 $\mathbb{E}_{y_t\sim\cdot}$ —— 在一个已经固定的前缀上，对词表求和时用谁的概率当权重。这一个位置上怎么打分。
- 外层：谁的权重在跑 generate，每个位置传回来的是：1 个标量还是 k 维向量
    - slime 的管道里这两件事是分开发生的，
        - 外层 = 学生。 rollout 阶段 SGLang 加载的是学生权重在采样。
        - 内层 = 学生（单样本 MC，$\hat D_t=\log q_t(y_t)-\log p_t(y_t)$）。
    - slime 教师只参与内层、不参与外层
        - teacher as reward_func: payload, "max_new_tokens": 0,   
  
序列级 forward KL 要求前缀 $y_{<t}$ 由**教师**生成 —— 即在教师的采样数据上做 SFT，即经典的 sequence-level KD，天然离线。这也正是 on-policy distillation 要解决的问题：学生在推理时访问的是自己的状态分布，否则在教师状态上训出来的模型碰到自己犯的错就不会往回走（exposure bias，即学生没在自己的错误状态上被训过）。

| | 外层：前缀 $y_{<t}$ 从谁采 | 内层：next token 期望对谁取 | slime 的管道满足吗 |
|---|---|---|---|
| Reverse KL | 学生 ✅（就是 on-policy rollout） | 学生 ✅（就是采出来的那个 token） | **两层都满足** |
| Forward KL | 教师 ❌ | 教师 ❌（要全分布） | 两层都不满足 |

slime 是纯 on-policy 管道 + 每 token 一个标量，对 forward KL 而言**两层都缺**。这就是它只能做 reverse KL 的完整原因——不是省事，是管道形状决定的。

顺带一个容易被忽略的点：verl 的 `forward_kl_topk` 它的前缀依然是学生 rollout，所以那个目标是"在学生访问的状态上，逐状态做 forward KL"——一个混合目标（GKD 那一系的做法），而不是序列级 forward KL。用它的时候心里要清楚你优化的到底是什么（“学生真的走到这里以后，教师下一步会怎么分配概率”。）。

### verl / slime

#### verl

- verl 的主线设计是 HybridFlow:
    - 单进程 controller 表达 RL 算法控制流,
    - 多进程 worker 承担 actor、 rollout、critic、reward model、teacher model 等重计算。
    - OPD 没有另起一个完全独立的 trainer,而是接入已有 PPO 主干

-------

> PPO、GRPO、OPD

PPO、GRPO、OPD 共享同一条 trainer 骨架: rollout，重算 old_log_probs，可选 ref/critic/reward，算 advantage，最后 update_actor。标准 PPO loss 是 ratio clipping:

$$
r_t = \exp(\log \pi_\theta(a_t) - \log \pi_{\text{old}}(a_t))
$$

$$
L = \max(-r_t A_t,\ -\operatorname{clip}(r_t, 1-\epsilon, 1+\epsilon)A_t)
$$

```python
# core_algos.py
pg_losses1 = -advantages * ratio
pg_losses2 = -advantages * torch.clamp(ratio, 1 - cliprange_low, 1 + cliprange_high)
clip_pg_losses1 = torch.maximum(pg_losses1, pg_losses2)
```

- 直觉是：如果 $A_t>0$，这个 token 比较好，模型会想增大它的概率，但 $r_t$ 超过 $1+\epsilon$ 后就不再给额外收益；如果 $A_t<0$，这个 token 比较差，模型会想降低它的概率，但 $r_t$ 低于 $1-\epsilon$ 后也不再允许继续从 loss 里获利。max 选的是“更保守、更大的 loss”，防止单步更新把策略推太远。
- 区别在于优化信号
    - PPO: adv_estimator=gae 时需要 critic value，advantage 来自 reward + value bootstrap；
    - GRPO: 不需要 critic，通常每个 prompt 采多条 response，在同组内按 outcome reward 做均值/方差归一化，再广播到 token；
        - PPO surrogate + group-relative advantage
    - OPD: reward 不是外部标量为主，而是 teacher 给的 token 级分布差异。若 use_task_rewards=False，普通 PPO/GRPO 的 task reward policy loss 会被置零，只保留 distillation term；
        - PPO trainer 基础设施 + teacher KL/logprob 监督

- k1 + use_policy_gradient=True
    - distillation_loss_mode=k1
    - use_policy_gradient=True
    - use_task_rewards=False
- $\delta_t(\theta)=\log\pi_\theta(Y_t\mid S_t)-\log p_T(Y_t\mid S_t).$
    - $A_t^{\mathrm{distill}}=-\operatorname{sg}\!\left(\operatorname{clip}(\delta_t)\right)$
 
```python
advantages=-distillation_losses.detach()
```

----

- 纯 OPD 配方甚至不使用任务 advantage
    - `use_task_rewards=False`

```python
policy_loss, policy_metrics = ppo_loss(...)
if not use_task_rewards:
    policy_loss = 0.0

policy_loss += distill_loss
```

#### slime

- slime/rollout/on_policy_distillation.py 里 reward_func 发给教师的 payload：
```python
payload = {
    "input_ids": sample.tokens,        # 学生的 prompt+response，整条作为教师的 *输入*
    "sampling_params": {
        "temperature": 0,
        "max_new_tokens": 0,           # ← 教师生成 0 个 token
        "skip_special_tokens": False,
    },
    "return_logprob": True,
    "logprob_start_len": 0,
}
```

- max_new_tokens: 0，教师一个 token 都不生成，它对学生已经写好的序列做一次 teacher-forcing 前向，逐位置吐出"你写的这个 token 我给多少分"（logits）。
    - input_ids = sample.tokens —— 学生的输出是教师的输入。
    - max_new_tokens: 0 —— 教师不采样 → 不参与外层。

```python
def apply_opd_kl_to_advantages(...):
    """Computes reverse KL (student_logp - teacher_logp) and adds weighted penalty
    to advantages in-place."""
    ...
    for i, adv in enumerate(advantages):
        reverse_kl = student_log_probs[i] - teacher_log_probs[i]
        advantages[i] = adv - args.opd_kl_coef * reverse_kl

```
$$
\hat{A}_t=A_t
-\lambda\left(\log \pi_\theta\left(y_t \mid y_{<t}\right)-\log \pi_{\mathrm{teacher}}\left(y_t \mid y_{<t}\right)\right), \qquad y_t \sim \pi_\theta
$$

- $A_t - \lambda\cdot\hat D_{KL}(\pi_{\text{student}}|\pi_{\text{teacher}})$，token 是学生自己采样出来的，log π_s − log π_t 在这个采样分布下的期望就是 reverse KL —— 标准的 k1 单样本估计。
- 教师那边拿回来的东西是 `on_policy_distillation.py` 从 SGLang 的 `meta_info["input_token_logprobs"]` 里抠的——只有教师在「学生已经采样出的那个 token」上的 logprob，一个标量。没有全词表分布，就只能构造 k1 这种单样本估计，算不了解析的 per-token KL，更算不了 forward KL（forward KL 要对教师分布求期望，得知道教师在所有 token 上的概率）。
- 另外注意这个惩罚项是 detached 常量（advantage 在训练前向之外算好），所以它是通过 policy gradient 的 score-function 形式回传的 —— 恰好是序列级 reverse KL 梯度的无偏估计，但方差比直接对 loss 求导大。

（一个实现细节：slime 的 OPD 项用的是**裸 k1**（`student_log_probs - teacher_log_probs`），逐 token 可正可负——某 token 学生给的概率低于教师时得到正的 advantage 奖励，期望上是对的。而它 `--kl-loss-type low_var_kl` 那个是 k3（Schulman 的控制变量版 $\,(r-1)-\log r$，恒非负、方差更低），管的是另一条常规 KL 支路，OPD 脚本里系数设成 0 关掉了。verl 那边 `loss_max_clamp=10.0` / `log_prob_min_clamp=-10.0` 就是给裸 k1 的负值和长尾兜底的。）

### review DPO

$$
\begin{split}
\mathcal{L}_{\mathrm{DPO}}(\theta)
&=-\mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}}
\left[\log \sigma\left(\beta \log
\frac{\pi_\theta(y_w\mid x)}
     {\pi_{\mathrm{ref}}(y_w\mid x)}
-\beta \log\frac{\pi_\theta(y_l\mid x)}{\pi_{\mathrm{ref}}(y_l\mid x)}\right)
\right]\\
&=-\mathbb E_{\mathcal D}\left[\log\sigma\left(\hat r_\theta(x,y_w)-\hat r_\theta(x,y_l)\right)\right]. \qquad \hat r_\theta(x,y)\coloneqq \beta\log\frac{\pi_\theta(y\mid x)}{\pi_{\mathrm{ref}}(y\mid x)}.
\end{split}
$$

$$
\nabla_\theta \mathcal{L}_{\mathrm{DPO}}
=-\beta \mathbb{E}_{\mathcal{D}}\left[
\underbrace{\sigma\!\left(
\hat{r}_\theta(x,y_l)-\hat{r}_\theta(x,y_w)
\right)}_{w_\theta:\,\text{排错越离谱权重越大}}
\left(\underbrace{\nabla_\theta \log \pi_\theta(y_w\mid x)
}_{\text{推高 chosen}}-\underbrace{\nabla_\theta \log \pi_\theta(y_l\mid x)
}_{\text{压低 rejected}}\right)
\right]
$$

- dpo vs. rl
    - 目标层面：DPO 的目标就是 RL 的目标。 在"BT 模型正确 + 数据无限 + 优化完美"的理想条件下，$\arg\min\mathcal{L}_{\text{DPO}}$ 与 RLHF 的最优解是同一个 $\pi^*$。DPO 没有换目标，只是换了一条到达目标的路——它把"先估 $r$、再用 RL 求 $\arg\max$"这个两阶段过程，用解析解短路成一步。
        - $\hat r_\theta(x,y)\coloneqq \beta\log\frac{\pi_\theta(y\mid x)}{\pi_{\mathrm{ref}}(y\mid x)}$
        - 它不是额外训练的奖励模型（reward model），而是由当前策略相对于参考策略的概率比推导出的隐式奖励：
    - 算法层面：DPO 不是 RL 算法。 它没有采样、没有探索（exploration）、没有环境交互、没有 credit assignment 的时序结构。它是一个在固定数据集上的最大似然估计（MLE），形式上是监督学习。RL 里"策略产生数据、数据改进策略"的闭环被彻底切断了。
    - 问题设定层面：这是 contextual bandit，不是完整 MDP。 DPO（以及 PPO-for-RLHF 的常规做法）把整条回复 $y$ 视作单个动作（single action），奖励在序列末端一次性给出，$\gamma=1$、单步。所以严格说这是上下文老虎机（contextual bandit）问题。
- 为什么 DPO 不是 on-policy
    - 最直接的层面：**数据分布固定不动**，策略在动。DPO 的数据 $(x, y_w, y_l)\sim\mathcal{D}$ 来自某个未知且固定的行为策略 $\mu$——可能是人写的、GPT 生成的、几个月前某个 checkpoint 采样的。训练过程中 $\pi_\theta$ 一直在变，$\mu$ 一步不动。分布错配（distribution shift）$\mathbb{D}_{\mathrm{KL}}(\pi_\theta|\mu)$ 随训练单调扩大，且这个偏移从未被算法测量、也从未被补偿。这甚至适用于"用 $\pi_{\text{ref}}$ 自己采样出来的偏好数据"这种最理想情况：只有第 1 个梯度步是 on-policy 的；从第 2 步起 $\pi_\theta \ne \pi_{\text{ref}}$，数据就已经是 off-policy 的了。"数据来自我自己的模型" $\ne$ on-policy——这是最常见的误解。
